# DARF — Dominance-Aware Regional LoRA Control

Bu notebook, **multi-LoRA dominansı** problemini sistematik olarak çözen DARF metodunun bütün bileşenlerini tek tek test eder.

## Problem
İki LoRA (Hermione + Daenerys) eşit α'da kullanılırsa modelin global UNet ağırlık uzayında baskın olan (Hermione) hep kazanıyor → iki Hermione çıkıyor. Sadece α'yı tune etmek yetmiyor çünkü LoRA residual her spatial konumda etkili.

## DARF Bileşenleri
1. **Aşama A — Dominance-aware adaptive control:** yanlış identity bir region'da kazanıyorsa onu bastır + doğru olanı yükselt. CLIP attribute scorer 3. feedback sinyali olarak.
2. **Aşama B — Spatially gated LoRA residuals (asıl katkı):** her UNet layer'da identity LoRA residual'ı kendi region mask'i ile çarpılıyor. Bunun üstüne layer-wise α (down/mid/up).
3. **Aşama C — Two-stage generation + face-local refinement:** önce kompozisyon, sonra kimlik. En son yüzü ayrı refine.

## Notebook Akışı
```
Bölüm 1: Setup (clone, LoRA indir, deps, cache)
Bölüm 2: Pipeline + scorers build
Bölüm 3: Baseline (eski adaptive loop) — referans
Bölüm 4: Test A — dominance + attribute
Bölüm 5: Test B — A + spatial LoRA gate + layer-wise α (asıl katkı)
Bölüm 6: Test C — two-stage (layout → identity)
Bölüm 7: Test D — face-local refinement (final cila)
Bölüm 8: Ablation tablosu — hepsini bir CSV'ye topla
```

## Bölüm 1 — Setup

RunPod'da ilk çalıştırmada repo klonlanır ve LoRA dosyaları Drive'dan indirilir. Sonraki run'larda `git pull` yeter.

In [ ]:
import os

REPO_URL = "https://github.com/melik3kara/LoRA-scale-adaptive-feedback.git"
REPO_DIR = "/workspace/LoRA-scale-adaptive-feedback"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"Repo already at {REPO_DIR}, pulling latest...")
    !cd {REPO_DIR} && git pull

In [ ]:
# LoRA dosyalarını Drive klasöründen indir (sadece ilk sefer)
!pip install -q gdown
os.makedirs(f"{REPO_DIR}/data/loras", exist_ok=True)

DRIVE_FOLDER_ID = "1L6NA_AT5HGSNOcKjw7gECbPKnaN86Ix5"

# Klasör zaten doluysa atla
import glob
if not glob.glob(f"{REPO_DIR}/data/loras/*.safetensors"):
    !gdown --folder "https://drive.google.com/drive/folders/{DRIVE_FOLDER_ID}" -O {REPO_DIR}/data/loras/
    import shutil
    for f in glob.glob(f"{REPO_DIR}/data/loras/**/*.safetensors", recursive=True):
        target = os.path.join(f"{REPO_DIR}/data/loras", os.path.basename(f))
        if f != target:
            shutil.move(f, target)

!ls -lh {REPO_DIR}/data/loras/

In [ ]:
# Cache + dependencies
os.environ["HF_HOME"] = "/workspace/cache/huggingface"
os.environ["INSIGHTFACE_HOME"] = "/workspace/cache/insightface"
os.makedirs("/workspace/cache/huggingface", exist_ok=True)
os.makedirs("/workspace/cache/insightface", exist_ok=True)

# Versiyon uyumluluğu için diffusers/transformers/peft pinli
!pip install -q diffusers==0.30.3 transformers==4.44.2 peft controlnet-aux insightface onnxruntime-gpu
!pip install -q "pydantic<2.10" "typing_extensions>=4.13.0"
!pip install -q mediapipe scipy   # Evaluator için


In [ ]:
os.chdir(REPO_DIR)
import sys
if "src" not in sys.path:
    sys.path.insert(0, "src")

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print("\nLoRAs:")
!ls -lh data/loras/
print("\nReference faces:")
!ls data/reference_faces/
print("\nPose images:")
!ls data/pose_images/

## Bölüm 2 — Pipeline + Scorer Build

Tek seferlik kurulum: SDXL+ControlNet+LoRA pipeline, FaceScorer (ArcFace), PoseScorer (OpenPose), AttributeScorer (CLIP).

**Önemli:** `target_keypoints` artık `OpenposeDetector` ile değil, `get_synthetic_target_keypoints()` ile alınıyor — synthetic skeleton'da detector çalışmıyor.

**Not:** İlk çalıştırmada SDXL (~7 GB) + ControlNet + InsightFace + CLIP indirilir, 5–10 dk sürer. Sonraki run'larda cache'den hızlı yüklenir.

In [ ]:
from pipeline import load_identities, build_pipeline
from scorer import FaceScorer
from pose_scorer import PoseScorer
from attribute_scorer import AttributeScorer
from evaluator import Evaluator
from generate_pose import get_synthetic_target_keypoints
from PIL import Image

identities = load_identities()
pipe = build_pipeline(identities)

pose_image = Image.open("data/pose_images/two_person_pose.png").convert("RGB")

face_scorer = FaceScorer(reference_dir="data/reference_faces", device="cuda")
pose_scorer = PoseScorer()
attr_scorer = AttributeScorer(identities, device="cuda")
evaluator   = Evaluator(device="cuda")

# Reference embeddings for Evaluator (Hungarian-assigned ArcFace)
ref_embeds = {k: evaluator.extract_face_embedding(
    Image.open(identities[k]["reference_image"]).convert("RGB")) for k in identities}

target_keypoints = get_synthetic_target_keypoints()

names = list(identities.keys())
identity_regions = {
    names[0]: (0,   0, 512,  1024),
    names[1]: (512, 0, 1024, 1024),
}

print(f"\nIdentities: {names}")
print(f"Identity regions: {identity_regions}")
print("Ready.")

### Evaluator yardımcı (her test cell'inin sonunda kullanılacak)

Hungarian-assigned ArcFace ve MediaPipe pose error'u tek satırda yazdırır. Diğer ekiple aynı metric'i raporlar.

In [ ]:
def show_evaluator_scores(label, image, pose_ref):
    sim_h, sim_d = evaluator.detect_and_assign_faces(
        image, ref_embeds["hermione"], ref_embeds["daenerys"]
    )
    pose_err = evaluator.pose_keypoint_error(image, pose_ref)
    print(f"[Evaluator/{label}] arcface_hermione={sim_h:.3f}  arcface_daenerys={sim_d:.3f}  pose_err={pose_err:.4f}")
    return {"label": label, "sim_h": sim_h, "sim_d": sim_d, "pose_err": pose_err}

## Bölüm 3 — Baseline (Eski Adaptive Loop)

Referans noktası: yeni mekanizmalar **kapalı**, sadece dominance signal kullanılıyor (eski davranış).

Bu cell'in çıktısı, ablation tablosunda `baseline` satırı olur. Beklenen: identity dominansı (iki Hermione veya Daenerys arcface çok düşük).

In [ ]:
from adaptive_loop import multi_lora_adaptive_generate

result_baseline = multi_lora_adaptive_generate(
    pipe, identities, pose_image,
    face_scorer, pose_scorer, target_keypoints, identity_regions,
    ctrl_scale=0.85,
    id_threshold=0.35,
    pose_threshold=0.50,
    alpha_init={"hermione": 0.30, "daenerys": 1.10},
    alpha_min ={"hermione": 0.05, "daenerys": 0.50},
    alpha_max ={"hermione": 0.50, "daenerys": 1.70},
    delta_up=0.10,
    delta_down=0.05,
    delta_suppress=0.0,                 # ← dominance suppression KAPALI
    attribute_scorer=None,              # ← attribute signal KAPALI
    block_profile=None,                 # ← layer-wise α KAPALI
    max_iters=5,
    seed=42,
    use_regional_attention=True,
    use_spatial_lora_gate=False,        # ← spatial gate KAPALI
)

print(f"\n[BASELINE] best_iter={result_baseline.best_iteration+1}")
print(f"final_alphas:  {result_baseline.final_alphas}")
print(f"final_arcface: {result_baseline.final_arcface}")
print(f"final_pose:    {result_baseline.final_pose}")
display(result_baseline.image.resize((512, 512)))

show_evaluator_scores("baseline", result_baseline.image, pose_image)


## Bölüm 4 — Test A: Dominance-aware Update + CLIP Attribute

**Açılan mekanizmalar:**
- `delta_suppress=0.07` — yanlış identity bir region'da kazanıyorsa onun α'sını DÜŞÜR + doğru olanı YÜKSELT.
- `attribute_scorer` — saç rengi / kıyafet uyumunu CLIP ile ölç. Arcface OK ama görsel yanlışsa α'yı bump et.

Akademik fikir: *"weak'i artır" yerine "baskını bastır + zayıfı artır"*.

Beklenti: Daenerys arcface 0.20+ aralığına çıkar, dominance < 0 olur.

In [ ]:
result_A = multi_lora_adaptive_generate(
    pipe, identities, pose_image,
    face_scorer, pose_scorer, target_keypoints, identity_regions,
    ctrl_scale=0.85,
    id_threshold=0.35,
    pose_threshold=0.50,
    alpha_init={"hermione": 0.30, "daenerys": 1.10},
    alpha_min ={"hermione": 0.05, "daenerys": 0.50},
    alpha_max ={"hermione": 0.50, "daenerys": 1.70},
    delta_up=0.10,
    delta_down=0.05,
    delta_suppress=0.07,                # ← AÇIK
    attribute_scorer=attr_scorer,       # ← AÇIK
    attribute_threshold=0.0,
    delta_attr=0.05,
    block_profile=None,                 # B kapalı
    max_iters=6,
    seed=42,
    use_regional_attention=True,
    use_spatial_lora_gate=False,        # B kapalı
)

print(f"\n[A] best_iter={result_A.best_iteration+1}")
print(f"final_alphas:  {result_A.final_alphas}")
print(f"final_arcface: {result_A.final_arcface}")
print(f"final_pose:    {result_A.final_pose}")
display(result_A.image.resize((512, 512)))

show_evaluator_scores("A", result_A.image, pose_image)


## Bölüm 5 — Test B: A + Spatially Gated LoRA + Layer-wise α (ASIL KATKI)

**Açılan mekanizmalar:**
- `use_spatial_lora_gate=True` — her UNet layer'da identity LoRA residual'ı kendi region mask'i ile çarpılıyor. Yani Hermione'nin LoRA residual'ı sağ region'da **sıfır**, Daenerys'inki sol region'da **sıfır**.
- `block_profile` — Hermione'yi early/composition layer'larında kıs (down=0.4), face-detail layer'larında (up=1.0) tut. Daenerys mid/up'ta güçlendir.

Akademik formül:
$$h_l(x) = W_l x + \sum_i \alpha_{i,l} \cdot M_i^{(l)} \odot \Delta W_{i,l} x$$

Bu paper'ın asıl novelty'si — sadece attention masking değil, **LoRA residual'ın kendisini bölgesel kısıtlamak**.

Beklenti: identity separation görsel olarak net, dominance ≤ 0, hem arcface hem attribute margin pozitif.

In [ ]:
block_profile = {
    "hermione": {"down": 0.4, "mid": 0.7, "up": 1.0},   # erken layer'larda zayıf
    "daenerys": {"down": 0.6, "mid": 1.1, "up": 1.3},   # mid/up'ta güçlü
}

result_B = multi_lora_adaptive_generate(
    pipe, identities, pose_image,
    face_scorer, pose_scorer, target_keypoints, identity_regions,
    ctrl_scale=0.85,
    id_threshold=0.35,
    pose_threshold=0.50,
    alpha_init={"hermione": 0.30, "daenerys": 1.10},
    alpha_min ={"hermione": 0.05, "daenerys": 0.50},
    alpha_max ={"hermione": 0.50, "daenerys": 1.70},
    delta_up=0.10,
    delta_down=0.05,
    delta_suppress=0.07,                # A açık
    attribute_scorer=attr_scorer,       # A açık
    block_profile=block_profile,        # ← B AÇIK (layer-wise α)
    max_iters=6,
    seed=42,
    use_regional_attention=True,
    use_spatial_lora_gate=True,         # ← B AÇIK (asıl katkı)
)

print(f"\n[B] best_iter={result_B.best_iteration+1}")
print(f"final_alphas:  {result_B.final_alphas}")
print(f"final_arcface: {result_B.final_arcface}")
print(f"final_pose:    {result_B.final_pose}")
display(result_B.image.resize((512, 512)))

show_evaluator_scores("B", result_B.image, pose_image)


## Bölüm 6 — Test C: Two-Stage Generation (Layout → Identity)

**Fikir:** Tek pass'te hem kompozisyonu hem kimliği çözmek zor. Bunu iki aşamaya böl:

1. **Stage 1 (layout):** düşük LoRA scale + yüksek ControlNet → poz, body placement, left/right ayrımı oturur. LoRA neredeyse devre dışı, model temiz bir kompozisyon çiziyor.
2. **Stage 2 (identity):** stage 1 image'i üzerine ControlNet img2img + yüksek asimetrik LoRA + spatial gate → faces/attire refine edilir, kompozisyon korunur (`strength=0.55` partial denoise).

Bu yöntem identity vs pose çatışmasını ayırıyor — pose stage 1'de çözülmüş olduğu için stage 2'de LoRA'ya "sadece kimlik" görevi kalıyor.

Beklenti: stage 1'de generic iki kişi, stage 2'de iki net farklı kimlik.

In [ ]:
from pipeline import generate_two_stage

stage1_img, stage2_img = generate_two_stage(
    pipe, identities, pose_image,
    layout_lora_scales={"hermione": 0.15, "daenerys": 0.40},     # düşük
    identity_lora_scales={                                         # yüksek + layer-wise
        "hermione": {"down": 0.15, "mid": 0.30, "up": 0.45},
        "daenerys": {"down": 0.60, "mid": 1.20, "up": 1.50},
    },
    layout_ctrl_scale=1.0,
    identity_ctrl_scale=0.75,
    refine_strength=0.55,
    refine_steps=28,
    layout_steps=28,
    seed=42,
    width=1024, height=1024,
    use_regional_attention=True,
    use_spatial_lora_gate=True,
    identity_regions=identity_regions,
)

print("\n[Stage 1 — layout]")
display(stage1_img.resize((512, 512)))
print("[Stage 2 — identity refinement]")
display(stage2_img.resize((512, 512)))

# Skorla
face_C  = face_scorer.score_image(stage2_img, identity_regions)
pose_C  = pose_scorer.score_image(stage2_img, target_keypoints, identity_regions)
attr_C  = attr_scorer.score_image(stage2_img, identity_regions)
for k in identities:
    print(f"{k}: arcface={face_C[k]['arcface']:.3f}  dom={face_C[k]['dominance']:+.3f}  "
          f"pose={pose_C[k]['pose']:.3f}  attr={attr_C[k]['margin']:+.3f}")

show_evaluator_scores("C", stage2_img, pose_image)


## Bölüm 7 — Test D: Face-local Refinement (Final Cila)

Yukarıdaki testlerden en iyi image üzerine çalıştır. Her yüz ayrı ayrı:
1. Crop (bbox + %50 padding)
2. 1024² upscale (SDXL native rez)
3. SDXL img2img — **sadece o identity'nin LoRA'sı aktif**, scale 1.4
4. Original boyuta downscale, feather edge ile geri yapıştır

Crop seviyesinde rakip LoRA olmadığı için her yüz tek geçişte doğru kimliğe yakınsıyor.

**Önemli:** Bu pipeline state'i değiştiriyor (set_adapters tek-tek aktif yapıyor). Sonrasında tekrar full-image üretim yapacaksan adapter'ları yeniden set et.

In [ ]:
from pipeline import face_local_refine

# Şimdiye kadar üretilen image'lardan en iyisini seç
candidates = {
    "baseline": result_baseline.image,
    "A":        result_A.image,
    "B":        result_B.image,
    "C":        stage2_img,
}
best_label = "B"   # genelde B veya C en iyi olur — gerekirse değiştir
best_input = candidates[best_label]
print(f"Refining best image ({best_label})...")

final_img = face_local_refine(
    pipe, identities, best_input,
    face_scorer, identity_regions,
    refine_strength=0.45,
    crop_pad_ratio=0.5,
    refine_steps=25,
    seed=42,
    feather=24,
)

print("\n[D — final cila]")
display(final_img.resize((512, 512)))

face_D = face_scorer.score_image(final_img, identity_regions)
attr_D = attr_scorer.score_image(final_img, identity_regions)
for k in identities:
    print(f"{k}: arcface={face_D[k]['arcface']:.3f}  dom={face_D[k]['dominance']:+.3f}  "
          f"attr={attr_D[k]['margin']:+.3f}")

show_evaluator_scores("D", final_img, pose_image)


## Bölüm 8 — Ablation Tablosu

Tüm konfigürasyonlardan rapor için CSV çıkar. Her satırda: arcface_self, dominance, pose, attribute margin.

Akademik raporda bu tabloya ek olarak:
- **Identity convergence rate** (max_iters içinde converged identity sayısı)
- **Mean iterations to convergence**
- **Inference time per iteration**

metrikleri ile zenginleştirebilirsin.

In [ ]:
import pandas as pd

def collect_metrics(label: str, image, regions):
    face = face_scorer.score_image(image, regions)
    pose = pose_scorer.score_image(image, target_keypoints, regions)
    attr = attr_scorer.score_image(image, regions)
    sim_h, sim_d = evaluator.detect_and_assign_faces(
        image, ref_embeds["hermione"], ref_embeds["daenerys"])
    pose_err = evaluator.pose_keypoint_error(image, pose_image)
    eval_sim = {"hermione": sim_h, "daenerys": sim_d}
    rows = []
    for k in identities:
        rows.append({
            "method":         label,
            "identity":       k,
            "arcface":        round(face[k]["arcface"], 4),
            "dominance":      round(face[k]["dominance"], 4),
            "pose":           round(pose[k]["pose"], 4),
            "attr_margin":    round(attr[k]["margin"], 4),
            "eval_arcface":   round(eval_sim[k], 4),
            "eval_pose_err":  round(pose_err, 4),
        })
    return rows

rows = []
rows += collect_metrics("baseline",            result_baseline.image, identity_regions)
rows += collect_metrics("A: dominance+attr",   result_A.image,        identity_regions)
rows += collect_metrics("B: A + spatial gate + layer-α", result_B.image, identity_regions)
rows += collect_metrics("C: two-stage",        stage2_img,            identity_regions)
rows += collect_metrics("D: C + face refine",  final_img,             identity_regions)

df = pd.DataFrame(rows)
os.makedirs("data/results/darf", exist_ok=True)
df.to_csv("data/results/darf/ablation.csv", index=False)
df

In [ ]:
# Image'ları diske kaydet (rapor görselleri için)
results_dir = "data/results/darf"
os.makedirs(results_dir, exist_ok=True)

result_baseline.image.save(f"{results_dir}/baseline.png")
result_A.image.save(f"{results_dir}/A_dominance_attr.png")
result_B.image.save(f"{results_dir}/B_spatial_gate.png")
stage1_img.save(f"{results_dir}/C_stage1_layout.png")
stage2_img.save(f"{results_dir}/C_stage2_identity.png")
final_img.save(f"{results_dir}/D_face_refined.png")

print(f"All images saved under {results_dir}/")
!ls -lh {results_dir}/

## Sonuçları İndir

Tüm image'ları ve `ablation.csv`'yi tek ZIP'te paketler — JupyterLab'da link'e tıklayınca yerele iner.
Sunum/rapor için image'ları buradan al.

In [ ]:
# === Tüm sonuçları ZIP'le ve indirilebilir link ver ===
import shutil
from IPython.display import FileLink

shutil.make_archive("/workspace/darf_v1_results", "zip", "data/results/darf")
print("Zip ready: /workspace/darf_v1_results.zip")
FileLink("/workspace/darf_v1_results.zip")

## Sonraki Adımlar

Bu notebook tek seed (42) ile çalışıyor. Akademik tablo için:

1. **Multi-seed sweep:** `from adaptive_loop import run_adaptive_best_of_n` ile her config'i 3-5 seed'le çalıştır, mean ± std raporla. Burada her config'i tek seed'le yapıyoruz.
2. **Failure rate metric:** Composite score < eşik → fail. N seed'in kaçı fail oluyor?
3. **Multi-pose ablation:** Farklı poz iskeletleriyle aynı tabloyu tekrarla — generalisation gösterilir.
4. **Alternatif base model:** `pipe = build_pipeline(identities, base_model="RunDiffusion/Juggernaut-XL-v9")` ile aynı tabloyu tekrarla — base model bağımsızlığı gösterilir.

## Akademik Çerçeve

> **Method: Dominance-Aware Regional LoRA Control (DARF).** We treat multi-LoRA inference as a dominance and spatial routing problem rather than a global scale-selection problem. Three mechanisms operate jointly: (1) a dominance-aware adaptive controller that boosts the correct identity *and* suppresses the wrong identity in any region where cross-identity similarity exceeds self-similarity; (2) layer-wise LoRA scaling that weakens dominant adapters in early UNet blocks (composition) while preserving them in late blocks (face detail); (3) **spatially gated LoRA residuals**, where each adapter's residual contribution at every UNet layer is multiplied by a spatial mask matching the adapter's identity region, downsampled to that layer's resolution. We additionally use a CLIP attribute scorer as a third feedback signal and a face-local refinement stage as a post-processing step.